# Validate database business rules

This notebook deliberately submits valid-looking and invalid transactions to PostgreSQL. Every test runs inside its own transaction and is rolled back, so the database keeps the same rows.

There are two result types:

- **PROTECTED**: PostgreSQL rejected a bad write as expected.
- **RULE GAP**: PostgreSQL accepted a write that the application says should be impossible. This is evidence that a trigger, controlled function, or transaction rule is still needed.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import text

project_root = Path(r"C:\Python\projects\pgSQL")
load_dotenv(project_root / ".env", override=True)
sys.path.insert(0, str(project_root / "src"))

from MAna.database import connect_postgres, execute_query

db = connect_postgres(driver="psycopg2")

[OK] Connected to database: postgresql+psycopg2://localhost/mansoura_mobility


## Test fixtures

The tests select existing keys from the generated dataset instead of assuming particular serial numbers. Negative keys are used for temporary rows, which also avoids consuming sequence values.


In [ ]:
fixture_query = """
SELECT
    (SELECT email FROM accounts WHERE email IS NOT NULL ORDER BY account_id LIMIT 1) AS existing_email,
    (SELECT passenger_id FROM passengers ORDER BY passenger_id LIMIT 1) AS passenger_id,
    (SELECT zone_id FROM zones ORDER BY zone_id LIMIT 1) AS pickup_zone_id,
    (SELECT zone_id FROM zones ORDER BY zone_id OFFSET 1 LIMIT 1) AS dropoff_zone_id,
    (SELECT offer_id FROM rides ORDER BY ride_id LIMIT 1) AS used_offer_id,
    (
        SELECT o.offer_id
        FROM offers o
        LEFT JOIN rides r ON r.offer_id = o.offer_id
        WHERE o.offer_status <> 'ACCEPTED' AND r.ride_id IS NULL
        ORDER BY o.offer_id
        LIMIT 1
    ) AS nonaccepted_offer_id,
    (
        SELECT r.ride_id
        FROM rides r
        LEFT JOIN passenger_driver_ratings pdr ON pdr.ride_id = r.ride_id
        WHERE pdr.ride_id IS NULL
        ORDER BY r.ride_id
        LIMIT 1
    ) AS unrated_ride_id,
    (
        SELECT r.ride_id
        FROM rides r
        LEFT JOIN passenger_driver_ratings pdr ON pdr.ride_id = r.ride_id
        WHERE r.ride_status = 'CANCELLED_BEFORE_START'
          AND pdr.ride_id IS NULL
        ORDER BY r.ride_id
        LIMIT 1
    ) AS cancelled_unrated_ride_id,
    (
        SELECT pa.payment_attempt_id
        FROM payment_attempts pa
        WHERE pa.payment_status = 'CAPTURED'
          AND pa.captured_amount_egp IS NOT NULL
        ORDER BY pa.payment_attempt_id
        LIMIT 1
    ) AS captured_payment_id,
    (
        SELECT pa.captured_amount_egp
        FROM payment_attempts pa
        WHERE pa.payment_status = 'CAPTURED'
          AND pa.captured_amount_egp IS NOT NULL
        ORDER BY pa.payment_attempt_id
        LIMIT 1
    ) AS captured_amount_egp,
    (SELECT ride_id FROM rides ORDER BY ride_id LIMIT 1) AS existing_ride_id
"""

fixtures = execute_query(fixture_query, db, return_results=True).iloc[0].to_dict()

vehicle_rows = execute_query(
    """
    SELECT driver_id, vehicle_id
    FROM vehicles
    ORDER BY driver_id, vehicle_id
    LIMIT 2
    """,
    db,
    return_results=True,
).to_dict("records")

if len(vehicle_rows) < 2 or vehicle_rows[0]["driver_id"] == vehicle_rows[1]["driver_id"]:
    vehicle_rows = execute_query(
        """
        SELECT DISTINCT ON (driver_id) driver_id, vehicle_id
        FROM vehicles
        ORDER BY driver_id, vehicle_id
        LIMIT 2
        """,
        db,
        return_results=True,
    ).to_dict("records")

fixtures.update(
    {
        "driver_1": vehicle_rows[0]["driver_id"],
        "vehicle_1": vehicle_rows[0]["vehicle_id"],
        "driver_2": vehicle_rows[1]["driver_id"],
        "vehicle_2": vehicle_rows[1]["vehicle_id"],
    }
)

pd.Series(fixtures, name="value").to_frame()

,value
existing_email,youssef.hamdy.1@example.test
passenger_id,1
pickup_zone_id,1
dropoff_zone_id,2
used_offer_id,1
nonaccepted_offer_id,20
unrated_ride_id,1
cancelled_unrated_ride_id,12
captured_payment_id,1
captured_amount_egp,30.0


## Transaction test helpers

Each case may contain one statement or several related statements. A database error counts as rejection; a successful set of statements counts as acceptance. Rollback happens in both cases.


In [ ]:
results = []


def check_case(rule, expected_result, statements):
    if isinstance(statements, tuple):
        statements = [statements]

    rejected = False
    error = ""

    with db.engine.connect() as connection:
        transaction = connection.begin()
        try:
            for sql, params in statements:
                connection.execute(text(sql), params)
        except Exception as exc:
            rejected = True
            error = str(getattr(exc, "orig", exc)).splitlines()[0][:180]
        finally:
            transaction.rollback()

    actual_result = "REJECTED" if rejected else "ACCEPTED"
    if expected_result == "REJECTED":
        classification = "PROTECTED" if rejected else "UNEXPECTED GAP"
    else:
        classification = "UNEXPECTED PROTECTION" if rejected else "RULE GAP"

    results.append(
        {
            "rule": rule,
            "expected test result": expected_result,
            "database result": actual_result,
            "classification": classification,
            "database message": error,
        }
    )

## Rules PostgreSQL already protects


In [ ]:
check_case(
    "Account email must be unique",
    "REJECTED",
    (
        """
        INSERT INTO accounts (account_id, full_name, email, account_status)
        VALUES (:account_id, :full_name, :email, 'ACTIVE')
        """,
        {"account_id": -900001, "full_name": "Constraint Test", "email": fixtures["existing_email"]},
    ),
)

check_case(
    "Passenger must reference an existing account",
    "REJECTED",
    (
        "INSERT INTO passengers (passenger_id) VALUES (:passenger_id)",
        {"passenger_id": -900002},
    ),
)

check_case(
    "Offer vehicle must belong to the stated driver",
    "REJECTED",
    (
        """
        INSERT INTO offers (
            offer_id, passenger_id, driver_id, vehicle_id,
            pickup_zone_id, dropoff_zone_id, initial_fare_egp,
            offer_status, initiated_at
        )
        VALUES (
            :offer_id, :passenger_id, :driver_id, :vehicle_id,
            :pickup_zone_id, :dropoff_zone_id, 80.00,
            'PENDING', CURRENT_TIMESTAMP
        )
        """,
        {
            "offer_id": -910001,
            "passenger_id": fixtures["passenger_id"],
            "driver_id": fixtures["driver_1"],
            "vehicle_id": fixtures["vehicle_2"],
            "pickup_zone_id": fixtures["pickup_zone_id"],
            "dropoff_zone_id": fixtures["dropoff_zone_id"],
        },
    ),
)

check_case(
    "Pickup and drop-off zones must differ",
    "REJECTED",
    (
        """
        INSERT INTO offers (
            offer_id, passenger_id, driver_id, vehicle_id,
            pickup_zone_id, dropoff_zone_id, initial_fare_egp,
            offer_status, initiated_at
        )
        VALUES (
            :offer_id, :passenger_id, :driver_id, :vehicle_id,
            :zone_id, :zone_id, 80.00, 'PENDING', CURRENT_TIMESTAMP
        )
        """,
        {
            "offer_id": -910002,
            "passenger_id": fixtures["passenger_id"],
            "driver_id": fixtures["driver_1"],
            "vehicle_id": fixtures["vehicle_1"],
            "zone_id": fixtures["pickup_zone_id"],
        },
    ),
)

check_case(
    "Initial offer fare must be greater than 25 EGP",
    "REJECTED",
    (
        """
        INSERT INTO offers (
            offer_id, passenger_id, driver_id, vehicle_id,
            pickup_zone_id, dropoff_zone_id, initial_fare_egp,
            offer_status, initiated_at
        )
        VALUES (
            :offer_id, :passenger_id, :driver_id, :vehicle_id,
            :pickup_zone_id, :dropoff_zone_id, 25.00,
            'PENDING', CURRENT_TIMESTAMP
        )
        """,
        {
            "offer_id": -910003,
            "passenger_id": fixtures["passenger_id"],
            "driver_id": fixtures["driver_1"],
            "vehicle_id": fixtures["vehicle_1"],
            "pickup_zone_id": fixtures["pickup_zone_id"],
            "dropoff_zone_id": fixtures["dropoff_zone_id"],
        },
    ),
)

check_case(
    "One offer can create at most one ride",
    "REJECTED",
    (
        """
        INSERT INTO rides (ride_id, offer_id, final_fare_egp, ride_status)
        VALUES (:ride_id, :offer_id, 80.00, 'AWAITING_PAYMENT')
        """,
        {"ride_id": -920001, "offer_id": fixtures["used_offer_id"]},
    ),
)

check_case(
    "Rating scores must stay between 1 and 5",
    "REJECTED",
    (
        """
        INSERT INTO passenger_driver_ratings (ride_id, attitude_score, submitted_at)
        VALUES (:ride_id, 6, CURRENT_TIMESTAMP)
        """,
        {"ride_id": fixtures["unrated_ride_id"]},
    ),
)

check_case(
    "A failed payment attempt must store a failure reason",
    "REJECTED",
    (
        """
        INSERT INTO payment_attempts (
            payment_attempt_id, ride_id, attempt_number, payment_method,
            payment_status, requested_amount_egp, provider_reference,
            initiated_at
        )
        VALUES (
            :payment_attempt_id, :ride_id, 999, 'CARD',
            'FAILED', 80.00, :provider_reference, CURRENT_TIMESTAMP
        )
        """,
        {
            "payment_attempt_id": -930001,
            "ride_id": fixtures["existing_ride_id"],
            "provider_reference": "ROLLBACK-FAILED-NO-REASON",
        },
    ),
)

check_case(
    "A completed refund must have a completion timestamp",
    "REJECTED",
    (
        """
        INSERT INTO refunds (
            refund_id, payment_attempt_id, refund_amount_egp,
            refund_reason, refund_status, requested_at
        )
        VALUES (
            :refund_id, :payment_attempt_id, 1.00,
            'Constraint test', 'COMPLETED', CURRENT_TIMESTAMP
        )
        """,
        {"refund_id": -940001, "payment_attempt_id": fixtures["captured_payment_id"]},
    ),
)

pd.DataFrame(results)

,rule,expected test result,database result,classification,database message
0,Account email must be unique,REJECTED,REJECTED,PROTECTED,duplicate key value violates unique constraint...
1,Passenger must reference an existing account,REJECTED,REJECTED,PROTECTED,"insert or update on table ""passengers"" violate..."
2,Offer vehicle must belong to the stated driver,REJECTED,REJECTED,PROTECTED,"insert or update on table ""offers"" violates fo..."
3,Pickup and drop-off zones must differ,REJECTED,REJECTED,PROTECTED,"new row for relation ""offers"" violates check c..."
4,Initial offer fare must be greater than 25 EGP,REJECTED,REJECTED,PROTECTED,"new row for relation ""offers"" violates check c..."
5,One offer can create at most one ride,REJECTED,REJECTED,PROTECTED,duplicate key value violates unique constraint...
6,Rating scores must stay between 1 and 5,REJECTED,REJECTED,PROTECTED,"new row for relation ""passenger_driver_ratings..."
7,A failed payment attempt must store a failure ...,REJECTED,REJECTED,PROTECTED,"new row for relation ""payment_attempts"" violat..."
8,A completed refund must have a completion time...,REJECTED,REJECTED,PROTECTED,"new row for relation ""refunds"" violates check ..."


## Business-rule gaps in the current schema

The next cases are expected to be accepted by the current schema even though the business model says they should not happen. That is not a failed test harness—it identifies the next database work.


In [ ]:
check_case(
    "A ride should only come from an accepted offer",
    "ACCEPTED",
    (
        """
        INSERT INTO rides (ride_id, offer_id, final_fare_egp, ride_status)
        VALUES (:ride_id, :offer_id, 80.00, 'AWAITING_PAYMENT')
        """,
        {"ride_id": -950001, "offer_id": fixtures["nonaccepted_offer_id"]},
    ),
)

pending_offer_sql = """
INSERT INTO offers (
    offer_id, passenger_id, driver_id, vehicle_id,
    pickup_zone_id, dropoff_zone_id, initial_fare_egp,
    offer_status, initiated_at
)
VALUES (
    :offer_id, :passenger_id, :driver_id, :vehicle_id,
    :pickup_zone_id, :dropoff_zone_id, 80.00,
    'PENDING', CURRENT_TIMESTAMP
)
"""

check_case(
    "A passenger should not have two open offers",
    "ACCEPTED",
    [
        (
            pending_offer_sql,
            {
                "offer_id": -950002,
                "passenger_id": fixtures["passenger_id"],
                "driver_id": fixtures["driver_1"],
                "vehicle_id": fixtures["vehicle_1"],
                "pickup_zone_id": fixtures["pickup_zone_id"],
                "dropoff_zone_id": fixtures["dropoff_zone_id"],
            },
        ),
        (
            pending_offer_sql,
            {
                "offer_id": -950003,
                "passenger_id": fixtures["passenger_id"],
                "driver_id": fixtures["driver_2"],
                "vehicle_id": fixtures["vehicle_2"],
                "pickup_zone_id": fixtures["pickup_zone_id"],
                "dropoff_zone_id": fixtures["dropoff_zone_id"],
            },
        ),
    ],
)

accepted_offer_sql = """
INSERT INTO offers (
    offer_id, passenger_id, driver_id, vehicle_id,
    pickup_zone_id, dropoff_zone_id, initial_fare_egp,
    offer_status, initiated_at, decided_at
)
VALUES (
    :offer_id, :passenger_id, :driver_id, :vehicle_id,
    :pickup_zone_id, :dropoff_zone_id, 80.00,
    'ACCEPTED', CURRENT_TIMESTAMP, CURRENT_TIMESTAMP
)
"""

ride_sql = """
INSERT INTO rides (ride_id, offer_id, final_fare_egp, ride_status)
VALUES (:ride_id, :offer_id, 80.00, :ride_status)
"""

check_case(
    "A passenger should not have two active rides",
    "ACCEPTED",
    [
        (
            accepted_offer_sql,
            {
                "offer_id": -950004,
                "passenger_id": fixtures["passenger_id"],
                "driver_id": fixtures["driver_1"],
                "vehicle_id": fixtures["vehicle_1"],
                "pickup_zone_id": fixtures["pickup_zone_id"],
                "dropoff_zone_id": fixtures["dropoff_zone_id"],
            },
        ),
        (
            accepted_offer_sql,
            {
                "offer_id": -950005,
                "passenger_id": fixtures["passenger_id"],
                "driver_id": fixtures["driver_2"],
                "vehicle_id": fixtures["vehicle_2"],
                "pickup_zone_id": fixtures["pickup_zone_id"],
                "dropoff_zone_id": fixtures["dropoff_zone_id"],
            },
        ),
        (ride_sql, {"ride_id": -960001, "offer_id": -950004, "ride_status": "AWAITING_PAYMENT"}),
        (ride_sql, {"ride_id": -960002, "offer_id": -950005, "ride_status": "AWAITING_PAYMENT"}),
    ],
)

check_case(
    "A ride should not start before payment authorization",
    "ACCEPTED",
    [
        (
            accepted_offer_sql,
            {
                "offer_id": -950006,
                "passenger_id": fixtures["passenger_id"],
                "driver_id": fixtures["driver_1"],
                "vehicle_id": fixtures["vehicle_1"],
                "pickup_zone_id": fixtures["pickup_zone_id"],
                "dropoff_zone_id": fixtures["dropoff_zone_id"],
            },
        ),
        (ride_sql, {"ride_id": -960003, "offer_id": -950006, "ride_status": "IN_PROGRESS"}),
    ],
)

check_case(
    "Total refunds should not exceed the captured payment",
    "ACCEPTED",
    (
        """
        INSERT INTO refunds (
            refund_id, payment_attempt_id, refund_amount_egp,
            refund_reason, refund_status, requested_at, completed_at
        )
        VALUES (
            :refund_id, :payment_attempt_id, :refund_amount_egp,
            'Over-refund test', 'COMPLETED', CURRENT_TIMESTAMP, CURRENT_TIMESTAMP
        )
        """,
        {
            "refund_id": -970001,
            "payment_attempt_id": fixtures["captured_payment_id"],
            "refund_amount_egp": float(fixtures["captured_amount_egp"]) + 1.0,
        },
    ),
)

check_case(
    "A ride cancelled before start should not be rateable",
    "ACCEPTED",
    (
        """
        INSERT INTO passenger_driver_ratings (
            ride_id, attitude_score, comfort_score, submitted_at
        )
        VALUES (:ride_id, 4, 4, CURRENT_TIMESTAMP)
        """,
        {"ride_id": fixtures["cancelled_unrated_ride_id"]},
    ),
)

## Validation report


In [ ]:
validation_report = pd.DataFrame(results)
validation_report

,rule,expected test result,database result,classification,database message
0,Account email must be unique,REJECTED,REJECTED,PROTECTED,duplicate key value violates unique constraint...
1,Passenger must reference an existing account,REJECTED,REJECTED,PROTECTED,"insert or update on table ""passengers"" violate..."
2,Offer vehicle must belong to the stated driver,REJECTED,REJECTED,PROTECTED,"insert or update on table ""offers"" violates fo..."
3,Pickup and drop-off zones must differ,REJECTED,REJECTED,PROTECTED,"new row for relation ""offers"" violates check c..."
4,Initial offer fare must be greater than 25 EGP,REJECTED,REJECTED,PROTECTED,"new row for relation ""offers"" violates check c..."
5,One offer can create at most one ride,REJECTED,REJECTED,PROTECTED,duplicate key value violates unique constraint...
6,Rating scores must stay between 1 and 5,REJECTED,REJECTED,PROTECTED,"new row for relation ""passenger_driver_ratings..."
7,A failed payment attempt must store a failure ...,REJECTED,REJECTED,PROTECTED,"new row for relation ""payment_attempts"" violat..."
8,A completed refund must have a completion time...,REJECTED,REJECTED,PROTECTED,"new row for relation ""refunds"" violates check ..."
9,A ride should only come from an accepted offer,ACCEPTED,ACCEPTED,RULE GAP,


In [ ]:
summary = (
    validation_report.groupby("classification", dropna=False)
    .size()
    .rename("tests")
    .reset_index()
)
summary

,classification,tests
0,PROTECTED,9
1,RULE GAP,6


## Interpretation

The protected rules need no immediate change. The rule gaps are the real result of this notebook. They should be handled later with targeted PostgreSQL triggers, controlled functions, or transaction logic—one rule at a time, with a repeatable test before and after each change.

This notebook proves write-time behavior. It does not prove that every existing row is realistic; that is a separate data-quality task.
